In [1]:
import pandas as pd
import numpy as np

In [2]:
# Remove the ID column
df_raw = pd.read_csv('./data/cirrhosis.csv').drop(columns=["ID", "N_Days"])

target_col = 'Status'

X = df_raw.drop(target_col, axis=1)
y = df_raw[target_col]

In [3]:
from sklearn.preprocessing import LabelEncoder

y = LabelEncoder().fit_transform(y)

### Manually splitting the dataset

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

print(f'Train lenght: {len(X_train)}')
print(f'Val lenght: {len(X_val)}')
print(f'Test lenght: {len(X_val)}')

Train lenght: 292
Val lenght: 63
Test lenght: 63


#### Pipelines

In [14]:
from sklearn.pipeline import Pipeline

In [15]:
num_cols = list(X.select_dtypes(include=['number']).columns)
cat_cols = list(X.select_dtypes(exclude=['number']).columns)

print(f'Num: {num_cols}')
print(f'Cat: {cat_cols}')

Num: ['Age', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin', 'Stage']
Cat: ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema']


##### Num 

In [16]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer, KNNImputer

num_ss_si_mean_pipeline = Pipeline([
    ('mean simple imputer', SimpleImputer(strategy='mean')),
    ('standard scaler', StandardScaler())
])

num_mm_si_mean_pipeline = Pipeline([
    ('mean simple imputer', SimpleImputer(strategy='mean')),
    ('minmax scaler', MinMaxScaler())
])

num_ss_knn_pipeline = Pipeline([
    ('knn imputer', KNNImputer()),
    ('standard scaler', StandardScaler())
])

num_mm_knn_pipeline = Pipeline([
    ('knn imputer', KNNImputer()),
    ('minmax scaler', MinMaxScaler())
])

##### Cat

In [17]:
from sklearn.preprocessing import OneHotEncoder

cat_si_unspec_ohe_pipeline = Pipeline([
    ('Unpecified simple imputer', SimpleImputer(strategy='constant', fill_value='Unspecified')),
    ('one hot encoder', OneHotEncoder(handle_unknown='ignore'))
])

#### Training & Evaluation

In [18]:
num_pipelines = {
    'Mean Simple Imputer + Standard Scaler': num_ss_si_mean_pipeline,
    'Mean Simple Imputer + MinMax Scaler': num_mm_si_mean_pipeline,
    'KNN Imputer + Standard Scaler': num_ss_knn_pipeline,
    'KNN Imputer + MinMax Scaler': num_mm_knn_pipeline
}

cat_pipelines = {
    'Constant Imputer (Unspecified) + OneHotEncoder': cat_si_unspec_ohe_pipeline
}

from prepare_models import create_default_models_dict

models = create_default_models_dict()

In [19]:
from utils import create_evaluation_dataframe

results_df = create_evaluation_dataframe(
    X_train,
    y_train,
    X_val,
    y_val,
    num_pipelines,
    cat_pipelines,
    models
)

results_df

,num_pipeline,cat_pipeline,model,train_accuracy,val_accuracy,train_precision,val_precision,train_recall,val_recall,train_f1,val_f1
0,KNN Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0000,0.7302,1.0000,0.6711,1.0000,0.7302,1.0000,0.6898
1,Mean Simple Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0000,0.7143,1.0000,0.6561,1.0000,0.7143,1.0000,0.6793
2,KNN Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.8048,0.7143,0.8162,0.6587,0.8048,0.7143,0.7822,0.6760
3,Mean Simple Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.8185,0.6984,0.8289,0.6442,0.8185,0.6984,0.7996,0.6580
4,Mean Simple Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0000,0.6984,1.0000,0.6378,1.0000,0.6984,1.0000,0.6623
5,KNN Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0000,0.6984,1.0000,0.6388,1.0000,0.6984,1.0000,0.6658
6,KNN Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.7363,0.6508,0.6942,0.6022,0.7363,0.6508,0.7049,0.6071
7,Mean Simple Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.7397,0.6508,0.6972,0.6022,0.7397,0.6508,0.7087,0.6071
8,KNN Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,DecisionTree,1.0000,0.6032,1.0000,0.5695,1.0000,0.6032,1.0000,0.5851
9,KNN Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,DecisionTree,1.0000,0.6032,1.0000,0.5718,1.0000,0.6032,1.0000,0.5869


In [20]:
mask_train = X_train[cat_cols].notna().all(axis=1)
mask_val = X_val[cat_cols].notna().all(axis=1)
mask_test = X_test[cat_cols].notna().all(axis=1)

X_train_clean = X_train[mask_train]
y_train_clean = y_train[mask_train]

X_val_clean = X_val[mask_val]
y_val_clean = y_val[mask_val]

X_test_clean = X_test[mask_test]
y_test_clean = y_test[mask_test]

cat_pipelines_clean = {
    'Dropped missing Cat + OneHotEncoder': Pipeline([
        ('one hot encoder', OneHotEncoder(handle_unknown='ignore'))
    ])
}

results_df_clean = create_evaluation_dataframe(
    X_train_clean,
    y_train_clean,
    X_val_clean,
    y_val_clean,
    num_pipelines,
    cat_pipelines_clean,
    models
)

results_df_clean

,num_pipeline,cat_pipeline,model,train_accuracy,val_accuracy,train_precision,val_precision,train_recall,val_recall,train_f1,val_f1
0,KNN Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,RandomForest,1.0000,0.7805,1.0000,0.6867,1.0000,0.7805,1.0000,0.7286
1,Mean Simple Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,RandomForest,1.0000,0.7561,1.0000,0.6669,1.0000,0.7561,1.0000,0.7044
2,Mean Simple Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,RandomForest,1.0000,0.7317,1.0000,0.6479,1.0000,0.7317,1.0000,0.6830
3,Mean Simple Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,SVC,0.8304,0.7317,0.8385,0.6426,0.8304,0.7317,0.8131,0.6824
4,KNN Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,SVC,0.8259,0.7317,0.8341,0.6426,0.8259,0.7317,0.8086,0.6824
5,KNN Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,RandomForest,1.0000,0.7317,1.0000,0.6426,1.0000,0.7317,1.0000,0.6824
6,Mean Simple Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,SVC,0.7768,0.6341,0.7356,0.5618,0.7768,0.6341,0.7516,0.5856
7,Mean Simple Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,DecisionTree,1.0000,0.6341,1.0000,0.6217,1.0000,0.6341,1.0000,0.6250
8,KNN Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,SVC,0.7723,0.6341,0.7308,0.5618,0.7723,0.6341,0.7474,0.5856
9,KNN Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,DecisionTree,1.0000,0.6341,1.0000,0.6165,1.0000,0.6341,1.0000,0.6225


In [23]:
df_imputed = results_df.copy()
df_dropped = results_df_clean.copy()

combined_results = pd.concat([df_imputed, df_dropped], ignore_index=True)

display(combined_results)

,num_pipeline,cat_pipeline,model,train_accuracy,val_accuracy,train_precision,val_precision,train_recall,val_recall,train_f1,val_f1
0,KNN Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0000,0.7302,1.0000,0.6711,1.0000,0.7302,1.0000,0.6898
1,Mean Simple Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0000,0.7143,1.0000,0.6561,1.0000,0.7143,1.0000,0.6793
2,KNN Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.8048,0.7143,0.8162,0.6587,0.8048,0.7143,0.7822,0.6760
3,Mean Simple Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.8185,0.6984,0.8289,0.6442,0.8185,0.6984,0.7996,0.6580
4,Mean Simple Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0000,0.6984,1.0000,0.6378,1.0000,0.6984,1.0000,0.6623
5,KNN Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0000,0.6984,1.0000,0.6388,1.0000,0.6984,1.0000,0.6658
6,KNN Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.7363,0.6508,0.6942,0.6022,0.7363,0.6508,0.7049,0.6071
7,Mean Simple Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.7397,0.6508,0.6972,0.6022,0.7397,0.6508,0.7087,0.6071
8,KNN Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,DecisionTree,1.0000,0.6032,1.0000,0.5695,1.0000,0.6032,1.0000,0.5851
9,KNN Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,DecisionTree,1.0000,0.6032,1.0000,0.5718,1.0000,0.6032,1.0000,0.5869


In [22]:
a = ['RandomForest', 'SVC', 'DecisionTree', 'GaussianNB']

for model in a:
    display(combined_results[combined_results['model'] == model])

,num_pipeline,cat_pipeline,model,train_accuracy,val_accuracy,train_precision,val_precision,train_recall,val_recall,train_f1,val_f1
0,KNN Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0,0.7302,1.0,0.6711,1.0,0.7302,1.0,0.6898
1,Mean Simple Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0,0.7143,1.0,0.6561,1.0,0.7143,1.0,0.6793
4,Mean Simple Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0,0.6984,1.0,0.6378,1.0,0.6984,1.0,0.6623
5,KNN Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0,0.6984,1.0,0.6388,1.0,0.6984,1.0,0.6658
16,KNN Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,RandomForest,1.0,0.7805,1.0,0.6867,1.0,0.7805,1.0,0.7286
17,Mean Simple Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,RandomForest,1.0,0.7561,1.0,0.6669,1.0,0.7561,1.0,0.7044
18,Mean Simple Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,RandomForest,1.0,0.7317,1.0,0.6479,1.0,0.7317,1.0,0.6830
21,KNN Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,RandomForest,1.0,0.7317,1.0,0.6426,1.0,0.7317,1.0,0.6824


,num_pipeline,cat_pipeline,model,train_accuracy,val_accuracy,train_precision,val_precision,train_recall,val_recall,train_f1,val_f1
2,KNN Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.8048,0.7143,0.8162,0.6587,0.8048,0.7143,0.7822,0.6760
3,Mean Simple Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.8185,0.6984,0.8289,0.6442,0.8185,0.6984,0.7996,0.6580
6,KNN Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.7363,0.6508,0.6942,0.6022,0.7363,0.6508,0.7049,0.6071
7,Mean Simple Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.7397,0.6508,0.6972,0.6022,0.7397,0.6508,0.7087,0.6071
19,Mean Simple Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,SVC,0.8304,0.7317,0.8385,0.6426,0.8304,0.7317,0.8131,0.6824
20,KNN Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,SVC,0.8259,0.7317,0.8341,0.6426,0.8259,0.7317,0.8086,0.6824
22,Mean Simple Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,SVC,0.7768,0.6341,0.7356,0.5618,0.7768,0.6341,0.7516,0.5856
24,KNN Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,SVC,0.7723,0.6341,0.7308,0.5618,0.7723,0.6341,0.7474,0.5856


,num_pipeline,cat_pipeline,model,train_accuracy,val_accuracy,train_precision,val_precision,train_recall,val_recall,train_f1,val_f1
8,KNN Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,DecisionTree,1.0,0.6032,1.0,0.5695,1.0,0.6032,1.0,0.5851
9,KNN Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,DecisionTree,1.0,0.6032,1.0,0.5718,1.0,0.6032,1.0,0.5869
10,Mean Simple Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,DecisionTree,1.0,0.5714,1.0,0.5657,1.0,0.5714,1.0,0.5681
11,Mean Simple Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,DecisionTree,1.0,0.5714,1.0,0.5597,1.0,0.5714,1.0,0.5639
23,Mean Simple Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,DecisionTree,1.0,0.6341,1.0,0.6217,1.0,0.6341,1.0,0.6250
25,KNN Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,DecisionTree,1.0,0.6341,1.0,0.6165,1.0,0.6341,1.0,0.6225
26,Mean Simple Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,DecisionTree,1.0,0.5854,1.0,0.5138,1.0,0.5854,1.0,0.5473
27,KNN Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,DecisionTree,1.0,0.5854,1.0,0.5341,1.0,0.5854,1.0,0.5548


,num_pipeline,cat_pipeline,model,train_accuracy,val_accuracy,train_precision,val_precision,train_recall,val_recall,train_f1,val_f1
12,Mean Simple Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,GaussianNB,0.1849,0.1905,0.6253,0.8354,0.1849,0.1905,0.1947,0.2123
13,Mean Simple Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,GaussianNB,0.1884,0.1905,0.5549,0.6590,0.1884,0.1905,0.1984,0.2267
14,KNN Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,GaussianNB,0.1610,0.1429,0.6859,0.2664,0.1610,0.1429,0.1632,0.1321
15,KNN Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,GaussianNB,0.1507,0.1270,0.3048,0.2490,0.1507,0.1270,0.1454,0.1110
28,Mean Simple Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,GaussianNB,0.2188,0.3171,0.6694,0.8481,0.2188,0.3171,0.2502,0.3389
29,Mean Simple Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,GaussianNB,0.2277,0.3171,0.6916,0.8481,0.2277,0.3171,0.2627,0.3389
30,KNN Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,GaussianNB,0.2232,0.3171,0.6623,0.8481,0.2232,0.3171,0.2532,0.3389
31,KNN Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,GaussianNB,0.2143,0.3171,0.6353,0.8481,0.2143,0.3171,0.2406,0.3389
